<a href="https://colab.research.google.com/github/roborew/ComputerVision/blob/main/MASK_RCNN_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.functional as F
from torchvision.models.detection import maskrcnn_resnet50_fpn
# from torchvision.models.detection import maskrcnn_resnet50_fpn_v2
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_Weights

import cv2
import numpy as np
import matplotlib.pyplot as plt
import gdown


AttributeError: module 'cv2.dnn' has no attribute 'DictValue'

https://pytorch.org/vision/main/models/mask_rcnn.html

COCO dataset, if we wanted to retrain

In [ ]:
# gdown.download("http://images.cocodataset.org/zips/val2017.zip", "val2017.zip")
# !unzip "val2017.zip" -d /content/data

# gdown.download("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", "annotations_trainval2017.zip")
# !unzip "annotations_trainval2017.zip" -d /content/data

Upload an image for using with the model.

In [ ]:
from google.colab import files
import matplotlib.pyplot as plt
uploaded = files.upload()

img_path = next(iter(uploaded))  # This gets the name of the first (and possibly only) uploaded file
# Read and display the image
img = plt.imread(img_path)
plt.imshow(img)
plt.axis('off')  # Optional: to not show axes for a cleaner image
plt.show()

Using pretrained maskrcnn

In [ ]:
# Load a pre-trained Mask R-CNN model
model = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
# model = maskrcnn_resnet50_fpn_v2(pretrained=True)

model = model.cuda()
model.eval()  # Set the model to evaluation mode

Convert image to tensor

In [ ]:
# Load an image
image = cv2.imread(img_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Convert the image to a tensor
image_tensor = torch.from_numpy(image / 255.).permute(2, 0, 1).float().unsqueeze(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
image_tensor = image_tensor.to(device)

In [ ]:
with torch.no_grad():
    predictions = model(image_tensor)

# # Extract masks for detected people
people_masks = [mask for mask, label in zip(predictions[0]['masks'], predictions[0]['labels']) if label == 1]

In [ ]:
blurred_image = image.copy()
CONFIDENCE = 0.9 # @param {type:"slider", min:0, max:1, step:0.1}
for mask in people_masks:
    binary_mask = mask[0] > CONFIDENCE  # Convert mask to binary
    for i in range(3):  # Iterate over each color channel
        # Apply blurring on the masked area
        blurred_image[:, :, i] = np.where(binary_mask.cpu().numpy(), cv2.GaussianBlur(blurred_image[:, :, i], (31, 31), 50), blurred_image[:, :, i])

In [ ]:
# Convert back to BGR for OpenCV
plt.imshow(blurred_image)
plt.axis('off')  # Optional: to not show axes for a cleaner image
plt.show()

Define class names and colours

In [ ]:
# Define a list of colors
colors = [
    "red", "green", "blue", "yellow", "purple", "orange", "lime", "cyan", "magenta", "pink",
    "navy", "maroon", "olive", "teal", "brown", "black", "gray", "violet", "gold", "silver",
]

COCO_INSTANCE_CATEGORY_NAMES = [
    'BG', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat',
    'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog',
    'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella',
    'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite',
    'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle',
    'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich',
    'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch',
    'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase',
    'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

class_colors = {cls: colors[i % len(colors)] for i, cls in enumerate(COCO_INSTANCE_CATEGORY_NAMES)}

Class detection with bounding box

In [ ]:
import torchvision.transforms as T
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import torch

# Predict and visualize
with torch.no_grad():
    predictions = model(image_tensor)

# Extract the bounding boxes, labels, and scores
boxes = predictions[0]['boxes']
labels = predictions[0]['labels']
scores = predictions[0]['scores']

# Convert the image tensor back to a PIL image for easier manipulation
image_np = image_tensor.squeeze().cpu().numpy()
image_np = np.transpose(image_np, (1, 2, 0))  # Change from (C, H, W) to (H, W, C)
image_np = (image_np * 255).astype(np.uint8)  # Scale back to [0-255] from [0-1]
image_pil = Image.fromarray(image_np)

CONFIDENCE_THRESHOLD = 0.7 # @param {type:"slider", min:0, max:1, step:0.1}

font_size = 100
font_path = '/usr/share/fonts/truetype/humor-sans/Humor-Sans.ttf'
font = ImageFont.truetype(font_path, font_size)

# Draw the bounding boxes and labels on the image
draw = ImageDraw.Draw(image_pil)
for box, label, score in zip(boxes, labels, scores):
    if score > CONFIDENCE_THRESHOLD:  # Only consider detections with a confidence score above a threshold (e.g., 50%)
        box = box.cpu().numpy()
        class_name = COCO_INSTANCE_CATEGORY_NAMES[label]
        color = class_colors.get(class_name, 'white')

        # Draw the box
        draw.rectangle([(box[0], box[1]), (box[2], box[3])], outline=color, width=3)

        # Annotate the box with the class name
        draw.text((box[0], box[1]), f"{class_name}: {score:.2f}", fill=color, font=font)

# Display the image with bounding boxes and labels
image_pil.show()
plt.axis('off')  # Optional: to not show axes for a cleaner image
# Convert back to BGR for OpenCV
plt.imshow(image_pil)



Table of preditions

In [ ]:
import pandas as pd

data = []
for box, label, score in zip(boxes, labels, scores):
    label_index = label.item()  # Adjust for zero-based indexing
    if label_index >= 0 and label_index < len(COCO_INSTANCE_CATEGORY_NAMES):
        class_name = COCO_INSTANCE_CATEGORY_NAMES[label_index]
    else:
        class_name = 'Unknown'  # Use a placeholder for out-of-range labels
    data.append({
        'Box (xmin, ymin, xmax, ymax)': f'({box[0]:.2f}, {box[1]:.2f}, {box[2]:.2f}, {box[3]:.2f})',
        'Label': class_name,
        'Score': f'{score:.2f}'
    })

df = pd.DataFrame(data)
df